# Workshop: Single Tool SRE Agent with Amazon Bedrock

## Overview

In this workshop module, you'll build an AI-powered Site Reliability Engineering (SRE) agent using Amazon Bedrock and the Strands framework. The agent will investigate Kubernetes pod failures and provide expert-level troubleshooting recommendations.

### Learning Objectives

By completing this module, you will:
- Configure Amazon Bedrock with Claude 3 Haiku for agent workloads
- Understand the Strands framework's @tool decorator pattern
- Build a FastAPI backend simulating infrastructure APIs
- Create an AI agent that performs systematic infrastructure troubleshooting
- Evaluate the performance benefits of AI-powered incident response

### Prerequisites

**AWS Requirements:**
- AWS Account with administrative access
- Amazon Bedrock access in your chosen region
- Claude 3 Haiku model enabled (us.anthropic.claude-3-haiku-20240307-v1:0)
- AWS CLI configured with appropriate credentials

**Technical Requirements:**
- Python 3.9+ environment
- Jupyter Notebook or JupyterLab
- Internet connectivity for package installation

**IAM Permissions Required:**
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": "arn:aws:bedrock:*::foundation-model/us.anthropic.claude-3-haiku-20240307-v1:0"
        }
    ]
}
```

### Architecture Overview

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│                 │    │                 │    │                 │
│ Strands Agent   │───▶│ @tool Function  │───▶│ FastAPI Backend │
│                 │    │                 │    │                 │
│ • Claude Haiku  │    │ get_pod_status  │    │ • Pod Data      │
│ • Investigation │    │                 │    │ • Realistic API │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

**Estimated completion time:** 20-25 minutes

## Step 1: Environment Validation and Setup

First, let's validate your environment and install required packages.

In [ ]:
# Environment validation
import sys
import subprocess
import json

def check_python_version():
    """Validate Python version meets requirements"""
    version = sys.version_info
    if version.major == 3 and version.minor >= 9:
        print(f"✅ Python {version.major}.{version.minor}.{version.micro} - Compatible")
        return True
    else:
        print(f"❌ Python {version.major}.{version.minor}.{version.micro} - Requires Python 3.9+")
        return False

def check_aws_credentials():
    """Validate AWS credentials are configured"""
    try:
        result = subprocess.run(['aws', 'sts', 'get-caller-identity'], 
                              capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            identity = json.loads(result.stdout)
            print(f"✅ AWS credentials configured for account: {identity.get('Account', 'Unknown')}")
            return True
        else:
            print(f"❌ AWS credentials not configured: {result.stderr}")
            return False
    except (subprocess.TimeoutExpired, FileNotFoundError, json.JSONDecodeError) as e:
        print(f"❌ AWS CLI not available or credentials not configured: {e}")
        return False

def get_aws_region():
    """Get current AWS region"""
    try:
        result = subprocess.run(['aws', 'configure', 'get', 'region'], 
                              capture_output=True, text=True, timeout=5)
        if result.returncode == 0 and result.stdout.strip():
            region = result.stdout.strip()
            print(f"✅ AWS region configured: {region}")
            return region
        else:
            print("⚠️  No default region configured, using us-east-1")
            return "us-east-1"
    except (subprocess.TimeoutExpired, FileNotFoundError) as e:
        print(f"⚠️  Cannot determine region, using us-east-1: {e}")
        return "us-east-1"

# Run validation checks
print("🔍 Environment Validation")
print("=" * 30)

python_ok = check_python_version()
aws_ok = check_aws_credentials()
aws_region = get_aws_region()

if not python_ok:
    raise RuntimeError("Python version requirement not met")
    
if not aws_ok:
    print("\n📋 AWS Setup Instructions:")
    print("1. Install AWS CLI: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html")
    print("2. Configure credentials: aws configure")
    print("3. Ensure Bedrock access in your region")
    raise RuntimeError("AWS credentials not configured")

print(f"\n✅ Environment validation complete - Region: {aws_region}")

In [ ]:
# Install required packages with version pinning for stability
%%bash
pip install --quiet \
    fastapi==0.104.1 \
    uvicorn[standard]==0.24.0 \
    strands==0.2.0 \
    requests==2.31.0 \
    boto3==1.34.0 \
    pydantic==2.5.0

echo "✅ Required packages installed successfully"

In [ ]:
# Import required libraries with error handling
import os
import time
import threading
import requests
from typing import Dict, Any, Optional

try:
    from fastapi import FastAPI, HTTPException
    from strands import Agent, tool
    from strands.models import BedrockModel
    import uvicorn
    print("✅ All libraries imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure all packages are installed correctly")
    raise

## Step 2: Create Infrastructure Simulation Backend

We'll create a FastAPI backend that simulates a Kubernetes cluster with realistic pod data representing common failure scenarios.

In [ ]:
# Create FastAPI application with comprehensive Kubernetes simulation
app = FastAPI(
    title="Kubernetes API Simulator",
    description="Simulates Kubernetes API for SRE agent training",
    version="1.0.0"
)

# Realistic pod data representing various failure scenarios
PODS_DATA = {
    "pods": [
        {
            "name": "payment-service-7d4f8-x5m1q",
            "namespace": "production",
            "status": "CrashLoopBackOff",
            "ready": False,
            "restart_count": 15,
            "cpu_usage": "25%",
            "memory_usage": "98%",
            "node": "worker-node-2",
            "last_restart": "2024-01-15T14:24:30Z",
            "age": "2h15m",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Waiting",
                    "reason": "CrashLoopBackOff",
                    "message": "Back-off 5m0s restarting failed container",
                    "exit_code": 137
                }
            ],
            "events": [
                "Warning: OutOfMemoryError in container payment-api",
                "Warning: Back-off restarting failed container",
                "Error: Container payment-api failed with exit code 137",
                "Warning: Failed to pull image payment-service:v1.2.3"
            ],
            "resource_limits": {
                "cpu": "500m",
                "memory": "512Mi"
            },
            "resource_requests": {
                "cpu": "250m",
                "memory": "256Mi"
            }
        },
        {
            "name": "user-service-9k2x1-y6n2r",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "32%",
            "memory_usage": "64%",
            "node": "worker-node-1",
            "last_restart": None,
            "age": "5d2h",
            "containers": [
                {
                    "name": "user-api",
                    "image": "user-service:v1.1.0",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully",
                    "exit_code": None
                }
            ],
            "events": [
                "Normal: Successfully pulled image user-service:v1.1.0",
                "Normal: Created container user-api",
                "Normal: Started container user-api"
            ],
            "resource_limits": {
                "cpu": "1000m",
                "memory": "1Gi"
            },
            "resource_requests": {
                "cpu": "500m",
                "memory": "512Mi"
            }
        },
        {
            "name": "database-service-abc123-def456",
            "namespace": "production",
            "status": "Pending",
            "ready": False,
            "restart_count": 0,
            "cpu_usage": "0%",
            "memory_usage": "0%",
            "node": None,
            "last_restart": None,
            "age": "10m",
            "containers": [
                {
                    "name": "postgres",
                    "image": "postgres:14",
                    "status": "Waiting",
                    "reason": "PodScheduled",
                    "message": "0/3 nodes are available: insufficient memory",
                    "exit_code": None
                }
            ],
            "events": [
                "Warning: FailedScheduling - 0/3 nodes are available: insufficient memory",
                "Normal: Scheduled - Successfully assigned to worker-node-3"
            ],
            "resource_limits": {
                "cpu": "2000m",
                "memory": "4Gi"
            },
            "resource_requests": {
                "cpu": "1000m",
                "memory": "2Gi"
            }
        }
    ]
}

@app.get("/health")
def health_check() -> Dict[str, str]:
    """Health check endpoint for the Kubernetes API simulator"""
    return {
        "status": "healthy",
        "service": "kubernetes-api-simulator",
        "version": "1.0.0"
    }

@app.get("/pods")
def get_pods(namespace: Optional[str] = None) -> Dict[str, Any]:
    """Get pods, optionally filtered by namespace"""
    if namespace:
        filtered_pods = [pod for pod in PODS_DATA["pods"] if pod["namespace"] == namespace]
        return {"pods": filtered_pods}
    return PODS_DATA

@app.get("/pods/{pod_name}")
def get_pod_details(pod_name: str) -> Dict[str, Any]:
    """Get detailed information about a specific pod"""
    for pod in PODS_DATA["pods"]:
        if pod["name"] == pod_name:
            return pod
    raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")

print("✅ FastAPI backend created with realistic Kubernetes data")
print(f"   - {len(PODS_DATA['pods'])} pods configured")
print("   - Multiple failure scenarios included")
print("   - Resource usage and events simulated")

In [ ]:
# Start FastAPI server with proper error handling
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"

def start_server():
    """Start FastAPI server in background thread"""
    try:
        uvicorn.run(
            app, 
            host=SERVER_HOST, 
            port=SERVER_PORT, 
            log_level="error",
            access_log=False
        )
    except Exception as e:
        print(f"❌ Server startup failed: {e}")

def wait_for_server(max_attempts: int = 10, delay: float = 1.0) -> bool:
    """Wait for server to become available"""
    for attempt in range(max_attempts):
        try:
            response = requests.get(f"{SERVER_URL}/health", timeout=2)
            if response.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(delay)
    return False

# Start server in background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server to be ready
print("🚀 Starting FastAPI server...")
if wait_for_server():
    # Verify server functionality
    try:
        health_response = requests.get(f"{SERVER_URL}/health", timeout=5)
        pods_response = requests.get(f"{SERVER_URL}/pods", timeout=5)
        
        if health_response.status_code == 200 and pods_response.status_code == 200:
            health_data = health_response.json()
            pods_data = pods_response.json()
            
            print(f"✅ Backend server running at {SERVER_URL}")
            print(f"   Health status: {health_data['status']}")
            print(f"   Pods available: {len(pods_data['pods'])}")
        else:
            print(f"❌ Server responding but with errors")
            print(f"   Health: {health_response.status_code}")
            print(f"   Pods: {pods_response.status_code}")
    except Exception as e:
        print(f"❌ Server verification failed: {e}")
else:
    print("❌ Server failed to start within timeout period")
    print("Please check for port conflicts or other issues")

## Step 3: Create Strands Agent Tool

Define a tool function that the AI agent can use to investigate Kubernetes pod issues.

In [ ]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get comprehensive status information for Kubernetes pods in the specified namespace.
    
    This tool queries the Kubernetes API simulator to retrieve detailed pod information
    including health status, resource usage, container details, and recent events.
    
    Args:
        namespace: Kubernetes namespace to query (default: production)
        
    Returns:
        Formatted string containing comprehensive pod status information including:
        - Pod health and readiness status
        - Resource usage (CPU and memory)
        - Container status and restart information
        - Recent events and error messages
        - Resource limits and requests
    """
    try:
        # Query the Kubernetes API simulator
        response = requests.get(
            f"{SERVER_URL}/pods",
            params={"namespace": namespace} if namespace != "production" else {},
            timeout=10
        )
        response.raise_for_status()
        data = response.json()
        
        # Filter pods by namespace
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        # Format comprehensive pod information
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            # Status indicators
            status_icon = "❌" if not pod["ready"] else "✅"
            
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Age: {pod.get('age', 'Unknown')}\n"
            result += f"   Node: {pod.get('node', 'Not scheduled')}\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            
            # Resource information
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            
            if pod.get('resource_limits'):
                limits = pod['resource_limits']
                result += f"   Resource Limits: CPU {limits.get('cpu', 'N/A')}, Memory {limits.get('memory', 'N/A')}\n"
            
            if pod.get('last_restart'):
                result += f"   Last Restart: {pod['last_restart']}\n"
            
            # Container details
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
                    if container.get('exit_code'):
                        result += f"       Exit Code: {container['exit_code']}\n"
                    if container.get('message'):
                        result += f"       Message: {container['message']}\n"
            
            # Recent events (show last 4 events for better context)
            if pod.get('events'):
                result += f"   Recent Events:\n"
                for event in pod['events'][-4:]:
                    result += f"     - {event}\n"
            
            result += "\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {str(e)}\nPlease ensure the backend server is running."
    except Exception as e:
        return f"Unexpected error occurred: {str(e)}"

print("✅ Strands tool function created: get_pod_status()")
print("   - Comprehensive pod status reporting")
print("   - Resource usage and limits included")
print("   - Container details and events captured")
print("   - Robust error handling implemented")

## Step 4: Initialize Amazon Bedrock Agent

Create a Strands Agent using Amazon Bedrock's Claude 3 Haiku model with professional SRE capabilities.

In [ ]:
# Initialize Bedrock model with error handling and validation
MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"

def validate_bedrock_access() -> bool:
    """Validate that Bedrock model is accessible"""
    try:
        # Create a test model instance
        test_model = BedrockModel(model_id=MODEL_ID, region=aws_region)
        
        # Try a simple test invocation
        test_response = test_model.invoke("Hello, this is a test. Please respond with 'OK'.")
        
        if test_response and "OK" in str(test_response):
            print(f"✅ Bedrock model {MODEL_ID} is accessible")
            return True
        else:
            print(f"⚠️  Bedrock model responded but may have issues")
            return True  # Continue anyway
            
    except Exception as e:
        print(f"❌ Bedrock model validation failed: {e}")
        return False

# Validate Bedrock access
print("🔍 Validating Amazon Bedrock Access")
print("=" * 35)

if not validate_bedrock_access():
    print("\n📋 Troubleshooting Steps:")
    print("1. Verify AWS credentials: aws sts get-caller-identity")
    print(f"2. Check Bedrock access in region: {aws_region}")
    print("3. Ensure Claude 3 Haiku model is enabled in Bedrock console")
    print("4. Verify IAM permissions for bedrock:InvokeModel")
    raise RuntimeError("Bedrock access validation failed")

try:
    # Create Bedrock model instance
    model = BedrockModel(model_id=MODEL_ID, region=aws_region)
    
    # Create Strands Agent with comprehensive SRE system prompt
    agent = Agent(
        model=model,
        tools=[get_pod_status],
        system_prompt="""You are an expert Site Reliability Engineer (SRE) with deep expertise in Kubernetes, 
        infrastructure troubleshooting, and incident response. Your role is to investigate production issues 
        systematically and provide actionable solutions.

        INVESTIGATION METHODOLOGY:
        1. Gather comprehensive data using available tools
        2. Analyze symptoms to identify root causes
        3. Assess impact and urgency levels
        4. Provide specific, prioritized remediation steps
        5. Suggest preventive measures and monitoring improvements

        ANALYSIS FOCUS AREAS:
        - Resource utilization patterns (CPU, memory, disk)
        - Container restart patterns and exit codes
        - Kubernetes events and error messages
        - Pod scheduling and node placement issues
        - Application-level vs infrastructure-level problems

        RESPONSE FORMAT:
        - Lead with immediate impact assessment
        - Provide confidence levels for your diagnoses
        - Include specific kubectl commands when relevant
        - Prioritize solutions by impact and implementation difficulty
        - Always include monitoring and prevention recommendations

        Be direct, technical, and solution-focused. Assume the audience has Kubernetes knowledge 
        but may need guidance on complex troubleshooting procedures."""
    )
    
    print(f"✅ Strands Agent initialized successfully")
    print(f"   Model: Claude 3 Haiku ({MODEL_ID})")
    print(f"   Region: {aws_region}")
    print(f"   Tools: 1 tool available (get_pod_status)")
    print(f"   Framework: Strands Agents")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    print("\n📋 Common Issues:")
    print("- Model not enabled in Bedrock console")
    print("- Insufficient IAM permissions")
    print("- Region mismatch")
    print("- Network connectivity issues")
    agent = None
    raise

## Step 5: Execute Production Incident Investigation

Simulate a critical production incident and let the AI agent investigate and provide recommendations.

In [ ]:
if agent:
    print("🚨 PRODUCTION INCIDENT SIMULATION")
    print("=" * 45)
    print("ALERT: Payment service experiencing issues")
    print("Impact: Users cannot complete purchases")
    print("Priority: P1 - Critical")
    print("Time: 2024-01-15 14:30:00 UTC")
    print("\n🤖 Initiating AI-powered investigation...\n")
    
    # Record investigation start time
    start_time = time.time()
    
    # Comprehensive incident description
    incident_description = """
    URGENT PRODUCTION INCIDENT:
    
    Our payment service is experiencing critical issues. Users are reporting they cannot 
    complete purchases, and our monitoring shows payment processing has dropped to zero 
    in the last 10 minutes.
    
    Initial symptoms:
    - Payment API returning 503 Service Unavailable
    - High error rates in application logs
    - Customer complaints increasing rapidly
    
    Please investigate the production environment immediately and:
    1. Identify what's wrong with the payment service
    2. Determine the root cause
    3. Provide specific steps to resolve the issue
    4. Suggest preventive measures
    
    This is a P1 incident affecting revenue - please prioritize accordingly.
    """
    
    try:
        # Execute investigation
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        
        print(f"⚡ Investigation completed in {investigation_time} seconds")
        print("\n" + "=" * 70)
        print("AI AGENT INVESTIGATION RESULTS")
        print("=" * 70)
        
        # Display response with proper formatting
        if hasattr(response, 'content'):
            if isinstance(response.content, list):
                for item in response.content:
                    if hasattr(item, 'text'):
                        print(item.text)
                    else:
                        print(str(item))
            else:
                print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        
        print("\n" + "=" * 70)
        
        # Store results for analysis
        investigation_results = {
            'duration': investigation_time,
            'response': response,
            'success': True
        }
        
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        print("\n📋 Possible causes:")
        print("- Model rate limiting")
        print("- Network connectivity issues")
        print("- Backend server not responding")
        
        investigation_results = {
            'duration': 0,
            'response': None,
            'success': False,
            'error': str(e)
        }
        
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    print("Please check the previous steps for errors")
    investigation_results = {'success': False, 'error': 'Agent not initialized'}

## Step 6: Performance Analysis and Validation

Analyze the AI agent's performance and validate its understanding of the infrastructure issues.

In [ ]:
if investigation_results.get('success'):
    print("📊 INVESTIGATION PERFORMANCE ANALYSIS")
    print("=" * 40)
    
    # Performance metrics
    duration = investigation_results['duration']
    print(f"\n⏱️  Performance Metrics:")
    print(f"   AI Investigation Time: {duration} seconds")
    print(f"   Manual Investigation (typical): 15-30 minutes")
    print(f"   Speed Improvement: ~{round((1800 - duration) / 1800 * 100)}% faster")
    
    # Capability assessment
    print(f"\n✅ Agent Capabilities Demonstrated:")
    print("   • Automatic tool selection and execution")
    print("   • Comprehensive data gathering")
    print("   • Root cause analysis")
    print("   • Prioritized remediation recommendations")
    print("   • Preventive measures suggestions")
    print("   • Professional SRE-level reasoning")
    
    # Business impact
    print(f"\n💼 Business Impact:")
    print("   • Reduced Mean Time To Resolution (MTTR)")
    print("   • 24/7 availability for incident response")
    print("   • Consistent investigation methodology")
    print("   • Reduced dependency on senior SRE availability")
    print("   • Improved incident documentation")
    
else:
    print("❌ Investigation failed - cannot perform analysis")
    if 'error' in investigation_results:
        print(f"Error: {investigation_results['error']}")

In [ ]:
# Validation test - specific technical question
if agent:
    print("🧪 TECHNICAL VALIDATION TEST")
    print("=" * 30)
    
    validation_prompt = """
    VALIDATION SCENARIO: A Kubernetes pod shows the following symptoms:
    - Status: CrashLoopBackOff
    - Memory usage: 98%
    - Exit code: 137
    - 15 restarts in the last hour
    
    What is the most likely root cause and what would be your first remediation step?
    """
    
    try:
        start_time = time.time()
        validation_response = agent(validation_prompt)
        validation_time = round(time.time() - start_time, 2)
        
        print(f"Validation completed in {validation_time} seconds\n")
        print("Agent Response:")
        print("-" * 15)
        
        # Display validation response
        if hasattr(validation_response, 'content'):
            if isinstance(validation_response.content, list):
                for item in validation_response.content:
                    if hasattr(item, 'text'):
                        response_text = item.text
                        print(response_text)
                    else:
                        response_text = str(item)
                        print(response_text)
            else:
                response_text = str(validation_response.content)
                print(response_text)
        else:
            response_text = str(validation_response)
            print(response_text)
        
        # Validate technical understanding
        key_concepts = [
            "memory", "OOM", "out of memory", "resource", "limit", 
            "137", "SIGKILL", "increase", "scale"
        ]
        
        found_concepts = [concept for concept in key_concepts 
                         if concept.lower() in response_text.lower()]
        
        print(f"\n📋 Technical Validation:")
        if len(found_concepts) >= 3:
            print(f"✅ Agent demonstrates strong technical understanding")
            print(f"   Key concepts identified: {', '.join(found_concepts[:5])}")
        else:
            print(f"⚠️  Agent response may lack technical depth")
            print(f"   Expected concepts: {', '.join(key_concepts[:5])}")
        
    except Exception as e:
        print(f"❌ Validation test failed: {e}")
else:
    print("❌ Agent not available for validation testing")

## Step 7: Cost and Usage Analysis

Analyze the cost implications and usage patterns of the AI agent.

In [ ]:
# Cost analysis for workshop participants
print("💰 COST ANALYSIS")
print("=" * 20)

# Estimated token usage for Claude 3 Haiku
estimated_input_tokens = 1500  # System prompt + user query
estimated_output_tokens = 800  # Agent response
total_tokens = estimated_input_tokens + estimated_output_tokens

# Claude 3 Haiku pricing (as of 2024)
input_cost_per_1k = 0.00025  # $0.25 per 1K input tokens
output_cost_per_1k = 0.00125  # $1.25 per 1K output tokens

input_cost = (estimated_input_tokens / 1000) * input_cost_per_1k
output_cost = (estimated_output_tokens / 1000) * output_cost_per_1k
total_cost = input_cost + output_cost

print(f"📊 Token Usage Estimate:")
print(f"   Input tokens: ~{estimated_input_tokens:,}")
print(f"   Output tokens: ~{estimated_output_tokens:,}")
print(f"   Total tokens: ~{total_tokens:,}")

print(f"\n💵 Cost Estimate (per investigation):")
print(f"   Input cost: ${input_cost:.6f}")
print(f"   Output cost: ${output_cost:.6f}")
print(f"   Total cost: ${total_cost:.6f}")

print(f"\n📈 Scale Projections:")
print(f"   100 investigations/month: ${total_cost * 100:.2f}")
print(f"   1,000 investigations/month: ${total_cost * 1000:.2f}")

print(f"\n⚖️  Cost vs. Benefit:")
print(f"   Average SRE hourly cost: $75-150")
print(f"   Time saved per incident: ~20 minutes")
print(f"   Labor cost saved: ${(20/60) * 100:.2f} per incident")
print(f"   ROI: ~{((20/60) * 100 / total_cost):.0f}x return on investment")

print(f"\n💡 Optimization Tips:")
print("   • Use Claude 3 Haiku for cost-effective operations")
print("   • Implement response caching for common issues")
print("   • Use streaming for real-time investigations")
print("   • Monitor token usage with CloudWatch")

## Step 8: Cleanup and Next Steps

Clean up resources and prepare for the next workshop module.

In [ ]:
# Store variables for next notebook
workshop_data = {
    'model_id': MODEL_ID,
    'aws_region': aws_region,
    'server_url': SERVER_URL,
    'pod_data_schema': PODS_DATA,
    'investigation_results': investigation_results
}

# Save to file for persistence
import json
with open('workshop_00_data.json', 'w') as f:
    json.dump({
        'model_id': MODEL_ID,
        'aws_region': aws_region,
        'server_url': SERVER_URL,
        'success': investigation_results.get('success', False)
    }, f, indent=2)

print("✅ Workshop data saved for next module")
print("\n📋 Cleanup Notes:")
print("   • FastAPI server will stop when notebook kernel stops")
print("   • No persistent AWS resources created")
print("   • Bedrock usage billed per token consumed")

print("\n🎯 Workshop Module Complete!")
print("=" * 30)
print("You have successfully:")
print("✅ Configured Amazon Bedrock with Claude 3 Haiku")
print("✅ Built a Strands Agent with custom tools")
print("✅ Simulated infrastructure troubleshooting")
print("✅ Analyzed AI agent performance and costs")
print("\n➡️  Ready for Workshop Module 1: Multi-Tool Agent")

## Summary and Key Takeaways

### What You Accomplished

In this workshop module, you successfully:

1. **Environment Setup**: Validated AWS credentials and Bedrock access
2. **Infrastructure Simulation**: Created a realistic Kubernetes API simulator
3. **AI Agent Development**: Built a Strands Agent with Amazon Bedrock
4. **Tool Integration**: Implemented the @tool decorator pattern
5. **Incident Response**: Demonstrated AI-powered troubleshooting
6. **Performance Analysis**: Measured speed and cost improvements

### Key Technical Learnings

- **Amazon Bedrock Integration**: Direct model access with AWS credentials
- **Strands Framework**: Simplified agent development with tool decorators
- **Claude 3 Haiku**: Cost-effective model for operational workloads
- **Tool Design**: Creating effective AI agent tools with proper error handling

### Business Value Demonstrated

- **Speed**: 95% faster incident investigation (seconds vs. minutes)
- **Consistency**: Systematic troubleshooting methodology
- **Availability**: 24/7 incident response capability
- **Cost-Effectiveness**: High ROI compared to manual investigation

### Production Considerations

For production deployment, consider:

**Security**:
- Implement proper authentication and authorization
- Use VPC endpoints for Bedrock access
- Encrypt data in transit and at rest
- Follow AWS security best practices

**Reliability**:
- Implement circuit breakers and retries
- Add comprehensive monitoring and alerting
- Design for graceful degradation
- Use multiple availability zones

**Scalability**:
- Implement response caching
- Use streaming for real-time responses
- Monitor token usage and costs
- Consider model fine-tuning for specific use cases

### Next Steps

Continue your learning journey with:

- **Module 1**: Multi-tool agent with expanded Kubernetes capabilities
- **Module 2**: Secure gateway integration with MCP protocol
- **Module 3**: Multi-agent architecture with specialist agents
- **Module 4**: Memory integration for persistent learning
- **Module 5**: Production deployment to AWS

### Resources

- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)
- [Strands Framework](https://strands.dev)
- [AWS Well-Architected Framework](https://aws.amazon.com/architecture/well-architected/)
- [Kubernetes Documentation](https://kubernetes.io/docs/)

---

**Congratulations!** You've completed the Single Tool SRE Agent workshop. You're now ready to build more sophisticated AI-powered infrastructure automation solutions.